# HDB Resale Candidate Training — Local / Colab Edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tengfone/hdb-resale-agentic-mlops/blob/main/hdb-resale-candidate-training-local-colab.ipynb)

Train an XGBoost regression model to predict Singapore HDB resale flat prices using the official data.gov.sg dataset (Jan 2017 onward).

This notebook runs entirely locally or in Google Colab — no AWS/SageMaker required. When the shared package is imported from a repo checkout, it also auto-loads a repo-local `.env` file if one exists.

**What this notebook does:**
1. Downloads the latest HDB resale dataset from the official data.gov.sg API.
2. Creates chronological train / validation / test splits.
3. Trains an XGBoost model inside a scikit-learn pipeline.
4. Evaluates the model with overall and segment-level metrics.
5. Logs the run to MLflow, defaulting to local SQLite when `MLFLOW_TRACKING_URI` is unset, and registers the model as a `candidate`.
6. Runs the agentic promotion workflow and persists the human review packet locally plus into MLflow when tracking is configured.


In [ ]:
# Install dependencies when running in Google Colab.
# If you are running locally, do this once before opening the notebook:
#   make install-notebook
# or
#   pip install -e '.[agent,notebook]'
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

in_colab = importlib.util.find_spec('google.colab') is not None

if in_colab:
    repo_dir = Path('/content/hdb-resale-agentic-mlops')
    bootstrap_marker = repo_dir / '.colab_bootstrap_complete'
    if not repo_dir.exists():
        subprocess.run(
            ['git', 'clone', 'https://github.com/tengfone/hdb-resale-agentic-mlops.git', str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    if not bootstrap_marker.exists():
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[agent,notebook]'],
            check=True,
        )
        bootstrap_marker.write_text('ok\n', encoding='utf-8')
        print(
            'Colab dependencies installed. Restarting the runtime once so '
            'NumPy and pandas reload cleanly. After the reconnect, run this '
            'cell again and continue with the notebook.'
        )
        sys.stdout.flush()
        os.kill(os.getpid(), 9)
    print(f'Colab bootstrap complete from {repo_dir}')
else:
    print(
        'Local environment detected. Ensure the repo venv already has '
        "`pip install -e '.[agent,notebook]'` before continuing."
    )


Bootstrap is handled by the cell above. In Colab, the first run will install dependencies and restart the runtime once before you continue.


In [ ]:
# Setup: imports and configuration
from __future__ import annotations

import json
import os
from datetime import datetime, timezone

import pandas as pd

from hdb_resale_mlops.config import ProjectPaths, RuntimeConfig
from hdb_resale_mlops.data import (
    chronological_split,
    load_or_download_snapshot,
    load_raw_resale_frame,
)
from hdb_resale_mlops.env import load_repo_env
from hdb_resale_mlops.evaluation import evaluate_model
from hdb_resale_mlops.features import (
    FEATURE_COLUMNS,
    build_model_hyperparameters,
    prepare_training_frame,
)
from hdb_resale_mlops.local_training import save_model, train_locally

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

env_file = load_repo_env()
paths = ProjectPaths.discover()
paths.ensure_local_dirs()
runtime_config = RuntimeConfig.from_env()
effective_mlflow_tracking_uri = runtime_config.mlflow_tracking_uri or 'sqlite:///mlflow.db'

run_label = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')
run_dir = paths.artifacts_dir / run_label
run_dir.mkdir(parents=True, exist_ok=True)

runtime_context = {
    'repo_root': str(paths.repo_root),
    'env_file': str(env_file) if env_file else None,
    'cache_dir': str(paths.cache_dir),
    'artifacts_dir': str(paths.artifacts_dir),
    'mlflow_tracking_uri': effective_mlflow_tracking_uri,
    'mlflow_experiment_name': runtime_config.mlflow_experiment_name,
    'mlflow_model_name': runtime_config.mlflow_model_name,
    'openai_model': os.getenv('OPENAI_MODEL', 'gpt-5-nano'),
    'market_research_provider': os.getenv('MARKET_RESEARCH_PROVIDER', 'auto'),
    'enable_judge_eval': os.getenv('ENABLE_JUDGE_EVAL', '').strip().lower() in {'1', 'true', 'yes', 'on'},
}
runtime_context


## Step 1 — Download the official data.gov.sg dataset

Downloads the full Jan 2017 onward HDB resale dataset via the data.gov.sg v1 download API and caches a CSV snapshot locally.

In [4]:
snapshot = load_or_download_snapshot(paths, runtime_config, force=False)
raw_frame = load_raw_resale_frame(snapshot)

print('snapshot path:', snapshot.csv_path)
print('source url:', snapshot.source_url)
print('pulled at:', snapshot.pulled_at)
print('rows:', len(raw_frame), '| reported total:', snapshot.record_count)
raw_frame.head()

snapshot path: /Users/tengfone/Repository/personal/mlops-agentic-eg/data/cache/hdb-resale-prices-2017-onward.csv
source url: https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view
pulled at: 2026-03-15T09:04:29.661473+00:00
rows: 226916 | reported total: 226916


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


## Step 2 — Build chronological splits and inspect features

The split uses the last 12 months for test, the previous 12 months for validation, and everything earlier for training.

In [5]:
split = chronological_split(
    raw_frame,
    validation_months=runtime_config.validation_months,
    test_months=runtime_config.test_months,
)

prepared_train = prepare_training_frame(split.train)
prepared_validation = prepare_training_frame(split.validation)
prepared_test = prepare_training_frame(split.test)

split.summary

{'train_rows': 175940,
 'validation_rows': 27345,
 'test_rows': 23631,
 'train_month_start': '2017-01',
 'train_month_end': '2024-03',
 'validation_month_start': '2024-04',
 'validation_month_end': '2025-03',
 'test_month_start': '2025-04',
 'test_month_end': '2026-03'}

In [6]:
feature_preview = prepared_train.loc[:, FEATURE_COLUMNS + ['resale_price']].head(10)
feature_preview

,town,flat_type,flat_model,storey_range,floor_area_sqm,flat_age_years,remaining_lease_years,storey_midpoint,resale_price
0,ANG MO KIO,2 ROOM,Improved,10 TO 12,44.0,38.0,61.333333,11.0,232000.0
1,ANG MO KIO,3 ROOM,New Generation,01 TO 03,67.0,39.0,60.583333,2.0,250000.0
2,ANG MO KIO,3 ROOM,New Generation,01 TO 03,67.0,37.0,62.416667,2.0,262000.0
3,ANG MO KIO,3 ROOM,New Generation,04 TO 06,68.0,37.0,62.083333,5.0,265000.0
4,ANG MO KIO,3 ROOM,New Generation,01 TO 03,67.0,37.0,62.416667,2.0,265000.0
5,ANG MO KIO,3 ROOM,New Generation,01 TO 03,68.0,36.0,63.000000,2.0,275000.0
6,ANG MO KIO,3 ROOM,New Generation,04 TO 06,68.0,38.0,61.500000,5.0,280000.0
7,ANG MO KIO,3 ROOM,New Generation,04 TO 06,67.0,41.0,58.333333,5.0,285000.0
8,ANG MO KIO,3 ROOM,New Generation,04 TO 06,68.0,38.0,61.500000,5.0,285000.0
9,ANG MO KIO,3 ROOM,New Generation,01 TO 03,67.0,38.0,61.333333,2.0,285000.0


## Step 3 — Train the model locally

Fits an XGBoost regressor inside a scikit-learn pipeline directly in this Python process (no SageMaker required).

In [7]:
hyperparameters = build_model_hyperparameters(random_seed=runtime_config.random_seed)
print('Hyperparameters:', json.dumps(hyperparameters, indent=2))

model, validation_evaluation = train_locally(
    split.train,
    split.validation,
    random_seed=runtime_config.random_seed,
)

model_path = save_model(model, run_dir / 'model.joblib')
print(f'Model saved to {model_path}')
print('Validation metrics:', validation_evaluation.overall_metrics)

Hyperparameters: {
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.05,
  "subsample": 0.8,
  "colsample_bytree": 0.8,
  "reg_lambda": 1.0,
  "min_child_weight": 1.0,
  "objective": "reg:squarederror",
  "random_state": 7,
  "n_jobs": -1
}
Model saved to /Users/tengfone/Repository/personal/mlops-agentic-eg/artifacts/20260405071910/model.joblib
Validation metrics: {'rmse': 125046.93181138101, 'mae': 103908.04307761931}


## Step 4 — Evaluate on the test set

Compute overall and segment-level metrics on the held-out test split.

In [8]:
test_evaluation = evaluate_model(model, prepared_test)

{
    'validation': validation_evaluation.overall_metrics,
    'test': test_evaluation.overall_metrics,
}

{'validation': {'rmse': 125046.93181138101, 'mae': 103908.04307761931},
 'test': {'rmse': 153527.73525157888, 'mae': 130641.85285641742}}

In [9]:
print('Validation by town')
display(validation_evaluation.segment_metrics['town'].tail(10))
print('Test by flat_type')
display(test_evaluation.segment_metrics['flat_type'].tail(10))

Validation by town


,segment,count,rmse,mae,mean_actual,mean_prediction
16,TAMPINES,1993,140345.579917,125843.804158,678216.207727,554146.5000
17,SERANGOON,434,142329.072648,113795.735239,683352.163594,574154.1250
18,SENGKANG,2113,143284.703344,124043.242006,645822.511122,522173.3125
19,BUKIT MERAH,1033,149782.325620,116526.477886,732067.627299,619089.3125
20,GEYLANG,695,150282.932852,124277.760063,626046.146619,503480.8125
21,QUEENSTOWN,695,151269.937310,120568.053192,703799.067626,584980.0625
22,KALLANG/WHAMPOA,925,155229.187231,125081.700034,700836.632432,577867.0625
23,BISHAN,377,158035.597684,135825.397878,830111.169761,695840.5000
24,TOA PAYOH,862,167353.923072,133886.201457,701084.276102,567379.0625
25,CENTRAL AREA,174,213829.427958,158174.267960,793009.856322,638152.0625


Test by flat_type


,segment,count,rmse,mae,mean_actual,mean_prediction
0,1 ROOM,6,21264.776784,20530.679688,2.657960e+05,2.521574e+05
1,2 ROOM,773,84364.352438,76873.697283,3.695352e+05,2.930417e+05
2,3 ROOM,5702,104826.270085,93307.624061,4.710359e+05,3.783513e+05
3,MULTI-GENERATION,5,141416.140168,126463.237500,1.139778e+06,1.013314e+06
4,4 ROOM,10304,164365.478469,143809.359064,6.739211e+05,5.306528e+05
5,5 ROOM,5426,174033.789625,146862.412283,7.855347e+05,6.398547e+05
6,EXECUTIVE,1415,180351.628572,152856.253952,9.291861e+05,7.808453e+05


## Step 5 — Log the run to MLflow and register the candidate model

Uses the configured `MLFLOW_TRACKING_URI` when present; otherwise this notebook falls back to a local SQLite-backed MLflow store.

To browse the logged runs after this cell completes:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```


In [10]:
from dataclasses import replace
from hdb_resale_mlops.mlflow_registry import log_and_register_candidate_model

mlflow_config = replace(runtime_config, mlflow_tracking_uri=effective_mlflow_tracking_uri)

training_job_metadata = {
    'training_mode': 'local',
    'model_path': str(model_path),
}

registration = log_and_register_candidate_model(
    model=model,
    validation_evaluation=validation_evaluation,
    test_evaluation=test_evaluation,
    runtime_config=mlflow_config,
    artifact_dir=run_dir / 'mlflow_artifacts',
    dataset_snapshot=snapshot.to_metadata(),
    split_summary=split.summary,
    hyperparameters=hyperparameters,
    training_job_metadata=training_job_metadata,
)
registration


/Users/tengfone/Repository/personal/mlops-agentic-eg/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/05 15:19:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/05 15:19:18 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY, TAVILY_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` 

RegistrationResult(run_id='ed542effa8f24c3bb532358737f93c21', model_name='hdb-resale-price-regressor', model_version='4', model_uri='models:/m-35d8c4b9588b4716b87dd77d2fdaea68')

## Results

In [11]:
result_summary = {
    'training_mode': 'local',
    'model_path': str(model_path),
    'mlflow_tracking_uri': effective_mlflow_tracking_uri,
    'mlflow_run_id': registration.run_id,
    'registered_model_name': registration.model_name,
    'registered_model_version': registration.model_version,
    'candidate_alias': 'candidate',
    'review_packets_dir': str(paths.artifacts_dir / 'promotion_reviews'),
    'judge_eval_enabled': os.getenv('ENABLE_JUDGE_EVAL', '').strip().lower() in {'1', 'true', 'yes', 'on'},
    'validation_metrics': validation_evaluation.overall_metrics,
    'test_metrics': test_evaluation.overall_metrics,
}
result_summary


{'training_mode': 'local',
 'model_path': '/Users/tengfone/Repository/personal/mlops-agentic-eg/artifacts/20260405071910/model.joblib',
 'mlflow_tracking_uri': 'sqlite:///mlflow.db',
 'mlflow_run_id': 'ed542effa8f24c3bb532358737f93c21',
 'registered_model_name': 'hdb-resale-price-regressor',
 'registered_model_version': '4',
 'candidate_alias': 'candidate',
 'review_packets_dir': '/Users/tengfone/Repository/personal/mlops-agentic-eg/artifacts/promotion_reviews',
 'judge_eval_enabled': True,
 'validation_metrics': {'rmse': 125046.93181138101, 'mae': 103908.04307761931},
 'test_metrics': {'rmse': 153527.73525157888, 'mae': 130641.85285641742}}

## Step 6 — Agentic Model Review & Promotion

The promotion workflow still uses the same three agentic design patterns, but the human step now writes a local review packet and mirrors it into MLflow so you can inspect or resume it later:

1. **Custom logic** — A deterministic policy engine gathers evidence (champion comparison, drift detection) and produces a verdict: `PROMOTE`, `REJECT`, or `MANUAL_REVIEW`.
2. **Single-agent (ReAct)** — An explainer agent with tools autonomously investigates the candidate model, querying metrics, comparing segments, and researching market trends with configurable Tavily or OpenAI-native web search. When `ENABLE_JUDGE_EVAL=true`, the workflow also logs advisory `judge_*` metrics and `judge_evaluation.json` without affecting the policy decision.
3. **Human-in-the-loop** — `start_promotion_review()` writes a review packet under `artifacts/promotion_reviews/`, mirrors it under `promotion_review/<review_id>/` in MLflow, and `resume_promotion_review()` completes the final approve/reject step later if needed.


In [12]:
from hdb_resale_mlops.promotion_workflow import start_promotion_review

candidate_segment_metrics = {
    col: df for col, df in test_evaluation.segment_metrics.items()
}

review = start_promotion_review(
    model_name=registration.model_name,
    model_version=registration.model_version,
    candidate_metrics=test_evaluation.overall_metrics,
    candidate_segment_metrics=candidate_segment_metrics,
    train_df=prepared_train,
    test_df=prepared_test,
)
review_id = review['review_id']

print(f"\nReview status: {review['status']}")
print(f"Policy verdict: {review['policy_verdict']['decision']}")
print(f"Review packet: {review['review_path']}")

if review.get('judge_evaluation'):
    print('Judge evaluation:')
    print(json.dumps(review['judge_evaluation'], indent=2))

print('\nReview report:')
print(review['report'])
print('\nRerun only Step 7 to record or change the human decision.')



Review status: pending_review
Policy verdict: MANUAL_REVIEW
Review packet: /Users/tengfone/Repository/personal/mlops-agentic-eg/artifacts/promotion_reviews/20260405T071918Z-hdb-resale-price-regressor-v4-promotion-1.json
Judge evaluation:
{
  "enabled": true,
  "status": "scored",
  "model": "qwen/qwen3.6-plus:free",
  "scores": {
    "completeness": 5,
    "accuracy": 3,
    "actionability": 5,
    "safety": 5,
    "average": 4.5,
    "reasoning": "Completeness: 5 - The report thoroughly covers all required dimensions: candidate/champion metrics, drift status, segment performance, and the MANUAL_REVIEW policy verdict. Accuracy: 3 - While all core metrics, drift statistics, and policy checks match the scenario context exactly, the report hallucinates training history for versions 1-4 and introduces external 2025 market data not provided in the prompt. P-values are also rounded to 0.0. Actionability: 5 - Highly actionable with a clear structure, specific risk flags, and concrete conditi

## Step 7 — Human Decision (Rerunnable)

This step loads the persisted review packet created above and records the human decision. Rerun this cell when you want to change only the final approve/reject action without regenerating the review.


In [13]:
from hdb_resale_mlops.promotion_workflow import load_promotion_review, resume_promotion_review

if 'review_id' not in globals():
    raise RuntimeError('Run Step 6 first so review_id is available.')

review = load_promotion_review(review_id)
print(f"Loaded review packet: {review['review_path']}")
print(f"Current review status: {review['status']}")

if review['status'] == 'completed':
    promotion_result = review
elif review['status'] == 'pending_review':
    print(review['report'])
    decision = input("\nApprove this model? [approve/reject]: ").strip().lower()
    if decision not in ('approve', 'reject'):
        print(f"Invalid input '{decision}', defaulting to 'reject'.")
        decision = 'reject'
    promotion_result = resume_promotion_review(review_id, decision)
elif review['status'] == 'auto_rejected':
    print(review['report'])
    decision = input(
        "\nModel was auto-rejected by policy. Type 'approve' to override and promote anyway, or press Enter to keep it rejected [approve/keep rejected]: "
    ).strip().lower()
    if decision not in ('', 'approve', 'reject'):
        print(f"Invalid input '{decision}', keeping the automatic rejection.")
        decision = ''
    promotion_result = (
        resume_promotion_review(review_id, decision)
        if decision in ('approve', 'reject')
        else review
    )
else:
    promotion_result = review

print(f"\nOutcome: {promotion_result['outcome']}")
print(f"Human decision: {promotion_result['human_decision']}")
print(f"Review packet: {promotion_result['review_path']}")


Loaded review packet: /Users/tengfone/Repository/personal/mlops-agentic-eg/artifacts/promotion_reviews/20260405T071918Z-hdb-resale-price-regressor-v4-promotion-1.json
Current review status: pending_review
## Summary
Candidate model v4 achieves an overall RMSE of 153,527.74 and MAE of 130,641.85, which is identical to the current champion model's metrics. The policy engine returned MANUAL_REVIEW due to data drift detected in four numeric features: floor_area_sqm (KS=0.0418), flat_age_years (KS=0.1322), remaining_lease_years (KS=0.1309), and storey_midpoint (KS=0.0114), all exceeding the 0.05 KS threshold. All other policy checks passed (absolute_rmse, absolute_mae, champion_rmse_regression, segment_rmse_regression). Segment-level performance shows 0.0% delta across all 26 towns and 7 flat types, confirming no degradation. However, the training history reveals that versions 1 through 4 all share identical test metrics, raising questions about whether meaningful model changes occurred bet

## Next steps

- Browse your MLflow runs: `mlflow ui --backend-store-uri sqlite:///mlflow.db`
- Copy `.env.example` to `.env` when you want persistent local MLflow, OpenAI, judge, or SageMaker-related settings.
- For the internal enterprise direct-job flow, refer to the MAESTRO internal docs.
- For the internal enterprise pipeline DAG flow, refer to the MAESTRO internal docs.


## Optional: expose the local MLflow UI from Google Colab

Use this only for development or testing. **This is insecure** because it exposes your MLflow server through a public tunnel. Do not use it for sensitive data, production tracking, or long-lived environments.

If you want to open the local `sqlite:///mlflow.db` UI from Colab, do this after the notebook run has produced local MLflow data:

1. In a **notebook cell**, get the runtime's public IP for the localtunnel password prompt:

```python
!curl ifconfig.me
```

2. Open the **Colab terminal** and install localtunnel:

```bash
npm install -g localtunnel
```

3. In the **same Colab terminal**, start MLflow:

```bash
mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --allowed-hosts "*" \
  --cors-allowed-origins "*"
```

4. Back in a **notebook cell**, start the tunnel:

```python
!lt --port 5000
```

5. Open the URL printed by localtunnel, and if prompted for a tunnel password, use the IP address from step 1.
